# Experiment 4: is the PDE loss a faithful target, and is the operator resolution-independent?

Four checks made while setting up experiments 1–3. Each one changed how the experiments were run or how their results should be read.

1. **Darcy residual.** The paper's finite-difference residual has a minimizer far from the true solution; a flux-conservative one doesn't.
2. **Kolmogorov PDE-loss grid.** How far the PDE loss's minimizer lies from the reference, on transient vs attractor states, at 64² and 128².
3. **Darcy fine-tuning floor.** Why all three optimizers ended at exactly the same error in experiment 2.
4. **Time-axis padding.** Why the Burgers operator is worse on the one time grid it never saw.

Everything here is regenerated from seeds (about 15 minutes on 2 CPU cores), except check 4, which loads the shipped Burgers weights `../weights/burgers_adam_s0.pt`.

Reference: Li, Zheng et al., *Physics-Informed Neural Operator for Learning Partial Differential Equations*, arXiv:2111.03794.

In [1]:
import os, sys, math, time
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, torch, pandas as pd
import pino_bench as pb, experiments as ex, analysis as an
torch.set_num_threads(2)

## 1. Darcy: which discrete residual?

The PDE loss is a residual evaluated on a grid. Its *minimizer* should be close to the true solution; otherwise, driving the loss to zero drives the network away from the truth.

Both residuals are linear in $u$, so we can compute the **exact least-squares minimizer** over the interior values (with $u=0$ on the boundary) and measure its distance to the reference solution, computed on a 241² grid and subsampled:

- **Paper's residual:** nested central differences, $-\big[(a u_x)_x + (a u_y)_y\big]$ with $u_x$ and $(\cdot)_x$ both central. It uses only interior rows, so there are fewer equations than unknowns, and it has a checkerboard null space.
- **Flux-conservative residual:** fluxes on cell faces with harmonic-mean coefficients. It gives a square, well-posed system.

In [2]:
rng = np.random.default_rng(1)
A = pb.grf_darcy_coeff(3, 241, rng=rng)
U = np.stack([pb.darcy_solve_one(x) for x in A])
a241, u241 = torch.tensor(A), torch.tensor(U)

def ls_minimizer_error(R, a, ut):
    s = a.shape[1]; n = s - 2
    apply = lambda V: R(torch.nn.functional.pad(V, (1, 1, 1, 1)), a.expand(V.shape[0], s, s)).reshape(V.shape[0], -1)
    I = torch.eye(n * n, dtype=torch.float64).reshape(n * n, n, n)
    M = torch.cat([apply(I[k:k + 500]) for k in range(0, n * n, 500)]).T
    sol = torch.linalg.lstsq(M, torch.ones(M.shape[0], 1, dtype=torch.float64), driver="gelsd").solution[:, 0]
    u = torch.nn.functional.pad(sol.reshape(1, n, n), (1, 1, 1, 1))
    return M.shape, 100 * pb.rel_l2(u, ut).item()

rows = []
for s in (31, 61):
    st = 240 // (s - 1)
    for b in range(3):
        a, ut = a241[b:b+1, ::st, ::st], u241[b:b+1, ::st, ::st]
        for name, R in [("paper (central differences)", pb.Darcy.residual), ("flux-conservative", pb.Darcy.residual_conservative)]:
            shape, err = ls_minimizer_error(R, a, ut)
            rows.append(dict(grid=s, sample=b, residual=name, equations=shape[0], unknowns=shape[1], minimizer_error_pct=round(err, 3)))
pd.DataFrame(rows).pivot_table(index=["grid", "sample"], columns="residual", values="minimizer_error_pct")

residual     flux-conservative  paper (central differences)
grid sample                                                
31   0                   1.327                       71.093
     1                   0.092                       70.614
     2                   0.490                       68.519
61   0                   0.604                       70.608
     1                   0.022                       70.179
     2                   0.196                       68.095

**Result.** The paper's residual has its exact minimizer about **70%** away from the true solution on both grids. Minimizing it pulls the network toward a wrong field: it has fewer equations than unknowns, and the checkerboard modes are invisible to nested central differences. The flux-conservative residual's minimizer is within about **1%**. Experiment 2 therefore uses the conservative form (`cfg["residual"] = "conservative"`; `"paper"` restores the original).

## 2. Kolmogorov flow: on which grid is the PDE loss faithful?

Same idea for the vorticity equation at $Re = 500$. Take a reference trajectory computed on 256², subsample it to a coarser grid, and **minimize the PDE loss directly over the space-time field**, with the initial frame fixed and L-BFGS. Then measure how far the minimizer moved from the reference.

The raw residual of the reference itself, relative to $\|f\|$, is also reported. It is *not* the right measure: the advection terms are much larger than $f$, so a large raw residual can coexist with a nearby minimizer.

Two reference states are used:
- **"transient":** after 3 time units of spin-up from a random field.
- **"attractor":** after 12 time units, on the chaotic attractor, where the flow is much rougher.

In [3]:
def reference_window(spin, seed, n=256, Re=500.0):
    cfg = dict(Re=Re, T=0.125, n_pde=1, n_data=0, n_val=1, n_test=0, train=dict(data_res=(n, 33)),
               gen=dict(n_gen=n, n_store=n, nt_store=33, substeps=4, n_traj=2, t_spin=spin, snap_every=0.2))
    return pb.make_kf_data(cfg, seed=seed, device="cpu", verbose=False)["w_eval"][0].double()

def drift(w_true, n, Re=500.0, iters=300):
    prob = pb.Kolmogorov.__new__(pb.Kolmogorov); prob.nu, prob.T = 1.0 / Re, 0.125
    wt = w_true[::w_true.shape[0] // n, ::w_true.shape[0] // n, :][None]
    w0 = wt[..., :1]; v = wt[..., 1:].clone().requires_grad_(True)
    Du, f = prob.residual(wt); r_truth = ((Du - f).norm() / f.norm()).item()
    opt = torch.optim.LBFGS([v], lr=1.0, max_iter=iters, history_size=50, tolerance_grad=1e-12,
                            tolerance_change=1e-15, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad(); Du, f = prob.residual(torch.cat([w0, v], -1)); loss = ((Du - f) ** 2).mean(); loss.backward(); return loss
    opt.step(closure)
    w = torch.cat([w0, v.detach()], -1)
    return r_truth, 100 * ((w - wt).norm() / wt.norm()).item()

t0 = time.time()
states = {"transient (t = 3)": reference_window(3.0, 0), "attractor (t = 12)": reference_window(12.0, 1)}
rows = []
for name, w in states.items():
    for n in (64, 128):
        r, d = drift(w, n)
        rows.append(dict(state=name, max_abs_w=round(w.abs().max().item(), 1), pde_grid=f"{n}² × 33",
                         raw_residual_rel_f=round(r, 3), minimizer_vs_reference_pct=round(d, 2)))
print(f"{time.time() - t0:.0f} s")
pd.DataFrame(rows)

285 s


,state,max_abs_w,pde_grid,raw_residual_rel_f,minimizer_vs_reference_pct
0,transient (t = 3),12.4,64² × 33,1.946,1.53
1,transient (t = 3),12.4,128² × 33,0.404,0.20
2,attractor (t = 12),19.6,64² × 33,8.085,9.01
3,attractor (t = 12),19.6,128² × 33,2.121,1.61


**Result.** On the attractor state the 64² PDE loss has its minimizer about **9%** from the reference, against about **1.6%** on 128². An earlier attractor sample gave 7.7% and 0.9%, so it varies by state, but 64² is consistently an order of magnitude worse. The milder transient state makes 64² look acceptable (≈1.5%), which is why a first version of these experiments used 64². The raw residual is large in every case, so it does not tell the grids apart; the minimizer distance does. Experiment 3 therefore puts the PDE loss on **128² × 33**. The paper used 256² × 65.

## 3. Darcy: why fine-tuning ended at the same error for every optimizer

In experiment 2, Adam, Muon and SOAP fine-tuned the same operator on five test instances. They followed visibly different paths but ended at identical errors. Here we regenerate those five test instances from the data seed and compute the **exact minimizer of the discrete (flux-conservative) PDE loss on the 61² grid**.

In [4]:
cfg = ex.cfg_for("darcy")
n_total = cfg["n_train"] + cfg["n_val"] + cfg["n_test"]
A = pb.grf_darcy_coeff(n_total, cfg["gen"]["s_gen"], rng=np.random.default_rng(2000))   # same stream as make_darcy_data
first_test = cfg["n_train"] + cfg["n_val"]
idx = list(range(first_test, first_test + cfg["finetune"]["n_instances"]))
floor = []
for i in idx:
    u_ref = torch.tensor(pb.darcy_solve_one(A[i]))[None, ::4, ::4]
    a = torch.tensor(A[i])[None, ::4, ::4]
    floor.append(ls_minimizer_error(pb.Darcy.residual_conservative, a, u_ref)[1])
ft = {o: [f["final_err"] * 100 for f in fs] for o, fs in an.ft_runs("darcy").items()}
df = pd.DataFrame({"instance": idx, "exact discrete minimizer (%)": floor, **{f"{an.NAME[o]} after fine-tuning (%)": v for o, v in ft.items()}})
df.round(4)

,instance,exact discrete minimizer (%),Adam after fine-tuning (%),Muon after fine-tuning (%),SOAP after fine-tuning (%)
0,1100,0.1061,0.1062,0.1062,0.1061
1,1101,0.0219,0.0219,0.0219,0.0219
2,1102,0.2558,0.2559,0.2559,0.2558
3,1103,0.0219,0.0219,0.0219,0.0219
4,1104,0.1873,0.1874,0.1874,0.1874


**Result.** All three optimizers end exactly at the error of the discrete loss's own minimizer. The fine-tuning problem is solved; what remains is the discretization error of the 61² scheme against the 241² reference. This is the paper's convergence study (Table 5) seen from the optimizer side: for a well-posed single-instance problem, final accuracy is set by the grid, and the optimizer only changes how fast you get there.

## 4. Burgers: discretization dependence along the padded time axis

In experiment 1 every optimizer was worse on the 64 × **51** test grid than on 32 × 26 or 128 × 101. Space is periodic, but time is not: the FNO zero-pads the time axis (Fourier continuation) by `ceil(0.25 · nt)` points before its FFTs.

Here we regenerate the first 100 test instances from the data seed, load the Adam operator from experiment 1 (seed 0), and vary the spatial and temporal grids separately.

In [5]:
cfg = ex.cfg_for("burgers"); g = cfg["gen"]
gen = torch.Generator().manual_seed(1000)              # same stream as make_burgers_data
first_test = cfg["n_train"] + cfg["n_val"]             # 900
batches = []
for i in range(0, first_test + 100, g["batch"]):
    batches.append(pb.grf_1d_periodic(min(g["batch"], cfg["n_train"] + cfg["n_val"] + cfg["n_test"] - i), g["nx_gen"], gen=gen))
u0 = torch.cat(batches)[first_test:first_test + 100]
t0 = time.time()
sol = pb.burgers_solve(u0, nu=cfg["nu"], dt=g["dt"], n_save=g["nt_store"])
stride = g["nx_gen"] // g["nx_store"]
prob = pb.make_problem("burgers", cfg, {"u0": u0[:, ::stride], "u": sol[:, ::stride, :], "split": (0, 0)})
model = prob.model(); model.load_state_dict(torch.load("../weights/burgers_adam_s0.pt", map_location="cpu")); model.eval()
test = torch.arange(100)
print(f"regenerated and solved in {time.time() - t0:.0f} s")
grid = {(nx, nt): round(100 * prob.evaluate(model, test, (nx, nt)), 3) for nx in (32, 64, 128) for nt in (26, 51, 101)}
pd.Series(grid).unstack().rename_axis(index="space points", columns="time points")

regenerated and solved in 13 s


time points,26,51,101
space points,,,
32,0.740,1.738,0.957
64,0.738,1.701,0.825
128,0.736,1.700,0.819


The error depends on the **time** grid only. Varying the padding at 51 time points, and predicting on 101 time points then subsampling to 51:

In [6]:
rows = []
for pf in (0.23, 0.25, 0.27):
    model.pad_frac = pf
    rows.append(dict(setting=f"51 time points, pad_frac {pf}", padded_length=51 + math.ceil(pf * 51),
                     error_pct=round(100 * prob.evaluate(model, test, (64, 51)), 3)))
model.pad_frac = 0.25
with torch.no_grad():
    u0c, u = prob.get(test, (64, 101))
    pred = prob.predict(model, u0c, 101)[:, :, ::2]
rows.append(dict(setting="predict on 101, subsample to 51", padded_length=101 + 26,
                 error_pct=round(100 * pb.rel_l2(pred, u[:, :, ::2]).item(), 3)))
pd.DataFrame(rows)

,setting,padded_length,error_pct
0,"51 time points, pad_frac 0.23",63,4.316
1,"51 time points, pad_frac 0.25",64,1.701
2,"51 time points, pad_frac 0.27",65,2.953
3,"predict on 101, subsample to 51",127,0.875


**Result.** The trained operator is accurate on the time grids it was trained on: 26 for the data loss and 101 for the PDE loss. On the unseen 51-point grid the error roughly doubles, and moving the padding by a single point changes it substantially. The same trained model gives 0.88% when evaluated at 101 points and subsampled to 51, so the solution is learned; it is the *discretization* the model is evaluated on that matters. Space, which is periodic and unpadded, shows no such effect.

This is a concrete instance of the debate on the "resolution-independent" slide (discretization convergence vs representation equivalence). The mechanism is not pinned down: candidates are the resolution-dependent length of the padded periodic box and aliasing from the pointwise nonlinearity along the padded axis. It also means **test grids that coincide with training grids can hide discretization dependence**. In the paper's Burgers setup, the "4×" grid is the PDE-loss grid.